In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_analysis_module import *
from association_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded mean conc command
Successfully loaded all modules


In [2]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_chimere(pol="ug_NO2", year=2019, SC="s1".upper())
# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod comname depcod depname  regcod           regname  \
0  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
1  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
2  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
3  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
4  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   

   age    pop2019    pop2030    pop2050  mort2019  mort2030  mort2050  \
0    0  31.979362  30.560224  28.974134  0.114652  0.101665  0.085457   
1    1  32.735555  30.384526  29.200275  0.018369  0.016510  0.014105   
2    2  33.511730  30.265005  29.417218  0.008333  0.007299  0.006310   
3    3  34.431724  30.206960  29.615683  0.005018  0.004605  0.004083   
4    4  35.587474  30.214303  29.787011  0.003853  0.003519  0.003135   

                                            geometry  
0  POLYGON ((497887

In [10]:
#-----------------------------------------------------------
# Codes to plot CHIMERE maps for PM2.5 and NO2 concentrations
#---------------------------------------------------------
import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

# Define scenarios, pollutants, and years
scenarios = ["s2", "s3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

OUTPUT_DIR = "data/2-output-data/maps_CHIMERE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Global settings for map plots
plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})


def pretty_pol(pol):
    return "PM2.5" if pol == "ug_PM25_RH50" else "NO2"


# -------------------------
# map plot function
# -------------------------
def fast_plot(ax, gdf, col, title, cmap, norm, pw_mean=None):
    xs = gdf.geometry.x.values
    ys = gdf.geometry.y.values

    # Extent with padding
    if len(xs) == 0 or len(ys) == 0:
        xmin, xmax, ymin, ymax = 0, 1, 0, 1
    else:
        xmin, xmax = np.nanmin(xs), np.nanmax(xs)
        ymin, ymax = np.nanmin(ys), np.nanmax(ys)
        pad_x = max((xmax - xmin) * 0.02, 1e-6)
        pad_y = max((ymax - ymin) * 0.02, 1e-6)
        xmin, xmax = xmin - pad_x, xmax + pad_x
        ymin, ymax = ymin - pad_y, ymax + pad_y

    # Scatter
    sc = ax.scatter(
        xs,
        ys,
        c=gdf[col].values,
        s=8,
        cmap=cmap,
        norm=norm,
        linewidths=0,
        alpha=0.95,
        rasterized=True
    )

    # Use Pop Weighted Mean instead of absolute mean in the title
    title_text = f"{title}"
    if pw_mean is not None:
        title_text += f"\n$\\mathbf{{Population\\ Weighted\\ Mean\\ =\\ {pw_mean:.2f}\\ \\mu g/m^3}}$"
    else:
        mean_val = np.nanmean(gdf[col].values)
        title_text += f"\nMean = {mean_val:.2f} µg/m³"

    ax.set_title(title_text, pad=10)

    # Geographic aspect ratio
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal', adjustable='box')

    # Keep proper Lat/Lon ticks
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.tick_params(axis='both', which='major', length=4, width=0.8, direction='out')

    # Cleaner spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    return sc


# -------------------------
# Main plotting function
# -------------------------
def plot_maps_for_pollutant_year(pol, year):
    pol_lbl = pretty_pol(pol)
    logging.info(f"Plotting {pol_lbl} - {year}")

    data = {}

    # ---- Prepare population data aggregated at commune level (comcod)
    pop_data = donnees_exportees_transformed.copy()
    pop_col = f"pop{year}"
    pop_data[pop_col] = pd.to_numeric(pop_data[pop_col], errors="coerce").fillna(0)

    # Aggregate population at commune level
    pop_commune = pop_data.groupby("comcod")[pop_col].sum().reset_index()
    pop_commune.rename(columns={pop_col: "pop_val"}, inplace=True)

    # ---- Load data once per scenario
    for sc in scenarios:
        logging.info(f"  Loading {sc.upper()}")
        scen = coordo_chimere(pol, year=year, SC=sc.upper())
        base = coordo_ineris_chimere(pol, year="2019")
        corr = correction_chimere(scen, base)

        # Helper to calculate pop weighted mean at commune level
        def get_pw_mean(target_gdf, val_col):
            # Spatial join to get comcod for grid points
            temp_gdf = gpd.sjoin(target_gdf, pop_data[["comcod", "geometry"]].drop_duplicates("comcod"), how="left",
                                 predicate="within")

            # Aggregate concentration/delta at commune level (simple mean of points within commune)
            commune_vals = temp_gdf.groupby("comcod")[val_col].mean().reset_index()

            # Merge with aggregated population
            merged = pd.merge(commune_vals, pop_commune, on="comcod", how="inner")

            total_pop = merged["pop_val"].sum()
            if total_pop == 0:
                return 0
            return (merged[val_col] * merged["pop_val"]).sum() / total_pop

        pw_mean_conc = get_pw_mean(scen, "conc")
        pw_mean_delta = get_pw_mean(corr, "delta_conc")

        data[sc] = {
            "scen": scen,
            "corr": corr,
            "pw_mean_conc": pw_mean_conc,
            "pw_mean_delta": pw_mean_delta
        }

    # ---- Shared color scales
    all_conc = np.concatenate([data[sc]["scen"]["conc"].values for sc in scenarios])
    all_delta = np.concatenate([data[sc]["corr"]["delta_conc"].values for sc in scenarios])

    conc_norm = Normalize(vmin=np.nanmin(all_conc), vmax=np.nanmax(all_conc))
    dmax = np.nanmax(np.abs(all_delta))
    delta_norm = TwoSlopeNorm(vcenter=0, vmin=-dmax, vmax=dmax)

    # ---- figure layout
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.subplots_adjust(left=0.09, right=0.95, top=0.88, bottom=0.06, wspace=0.00, hspace=0.50)
    fig.suptitle(f"{pol_lbl} — {year} (S2 and S3)", fontsize=18)

    for i, sc in enumerate(scenarios):
        scen = data[sc]["scen"]
        corr = data[sc]["corr"]

        # Concentration map
        sc_plot = fast_plot(
            axes[i, 0],
            scen,
            "conc",
            f"{sc.upper()} — Average Concentration",
            "viridis",
            conc_norm,
            pw_mean=data[sc]["pw_mean_conc"]
        )

        # Delta map
        delta_plot = fast_plot(
            axes[i, 1],
            corr,
            "delta_conc",
            f"{sc.upper()} — Δ Concentration",
            "coolwarm",
            delta_norm,
            pw_mean=data[sc]["pw_mean_delta"]
        )

        # ---- Proper publication-style colorbars
        cbar_ax1 = axes[i, 0].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb1 = plt.colorbar(sc_plot, cax=cbar_ax1)
        cb1.set_label("Mean Conc (µg/m³)", fontsize=12)
        cb1.ax.tick_params(labelsize=14)

        cbar_ax2 = axes[i, 1].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb2 = plt.colorbar(delta_plot, cax=cbar_ax2)
        cb2.set_label("Δ Conc (µg/m³)", fontsize=14)
        cb2.ax.tick_params(labelsize=12)

    # ---- Save
    out_path = os.path.join(OUTPUT_DIR, f"pop_weighted_{pol_lbl}_{year}_S2_and_S3.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)

    logging.info(f"Saved: {out_path}")


# -------------------------
# Main driver
# -------------------------
if __name__ == "__main__":
    for pol in pollutants:
        for year in years:
            plot_maps_for_pollutant_year(pol, year)

    logging.info("✅ All maps generated with population-weighted formatting.")


2026-01-18 19:52:49,951 - INFO - Plotting PM2.5 - 2030
2026-01-18 19:52:49,984 - INFO -   Loading S2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS2_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:52:51,411 - INFO -   Loading S3


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS3_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:52:57,246 - INFO - Saved: data/2-output-data/maps_CHIMERE\pop_weighted_PM2.5_2030_S2_and_S3.png
2026-01-18 19:52:57,246 - INFO - Plotting PM2.5 - 2050
2026-01-18 19:52:57,265 - INFO -   Loading S2


...

2026-01-18 19:53:33,157 - INFO - Saved: data/2-output-data/maps_CHIMERE\pop_weighted_NO2_2030_S2_and_S3.png
2026-01-18 19:53:33,157 - INFO - Plotting NO2 - 2050
2026-01-18 19:53:33,186 - INFO -   Loading S2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS2_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:53:36,238 - INFO -   Loading S3


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS3_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:53:51,324 - INFO - Saved: data/2-output-data/maps_CHIMERE\pop_weighted_NO2_2050_S2_and_S3.png
2026-01-18 19:53:51,324 - INFO - ✅ All maps generated with population-weighted formatting.


In [11]:
#-----------------------------------------------------------
# Codes to plot CHIMERE maps for PM2.5 and NO2 concentrations
#---------------------------------------------------------
import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

# Define scenarios, pollutants, and years
scenarios = ["s2", "s3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

OUTPUT_DIR = "data/2-output-data/maps_CHIMERE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Global settings for map plots
plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})


def pretty_pol(pol):
    return "PM2.5" if pol == "ug_PM25_RH50" else "NO2"


# -------------------------
# map plot function
# -------------------------
def fast_plot(ax, gdf, col, title, cmap, norm):
    xs = gdf.geometry.x.values
    ys = gdf.geometry.y.values

    # Extent with padding
    if len(xs) == 0 or len(ys) == 0:
        xmin, xmax, ymin, ymax = 0, 1, 0, 1
    else:
        xmin, xmax = np.nanmin(xs), np.nanmax(xs)
        ymin, ymax = np.nanmin(ys), np.nanmax(ys)
        pad_x = max((xmax - xmin) * 0.02, 1e-6)
        pad_y = max((ymax - ymin) * 0.02, 1e-6)
        xmin, xmax = xmin - pad_x, xmax + pad_x
        ymin, ymax = ymin - pad_y, ymax + pad_y

    # Scatter
    sc = ax.scatter(
        xs,
        ys,
        c=gdf[col].values,
        s=8,
        cmap=cmap,
        norm=norm,
        linewidths=0,
        alpha=0.95,
        rasterized=True
    )

    mean_val = np.nanmean(gdf[col].values)
    title_text = f"{title}\nMean = {mean_val:.2f} µg/m³"

    ax.set_title(title_text, pad=10)

    # Geographic aspect ratio
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal', adjustable='box')

    # Keep proper Lat/Lon ticks
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.tick_params(axis='both', which='major', length=4, width=0.8, direction='out')

    # Cleaner spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    return sc


# -------------------------
# Main plotting function
# -------------------------
def plot_maps_for_pollutant_year(pol, year):
    pol_lbl = pretty_pol(pol)
    logging.info(f"Plotting {pol_lbl} - {year}")

    data = {}

    # ---- Load data once per scenario
    for sc in scenarios:
        logging.info(f"  Loading {sc.upper()}")
        scen = coordo_chimere(pol, year=year, SC=sc.upper())
        base = coordo_ineris_chimere(pol, year="2019")
        corr = correction_chimere(scen, base)

        data[sc] = {
            "scen": scen,
            "corr": corr
        }

    # ---- Shared color scales
    all_conc = np.concatenate([data[sc]["scen"]["conc"].values for sc in scenarios])
    all_delta = np.concatenate([data[sc]["corr"]["delta_conc"].values for sc in scenarios])

    conc_norm = Normalize(vmin=np.nanmin(all_conc), vmax=np.nanmax(all_conc))
    dmax = np.nanmax(np.abs(all_delta))
    delta_norm = TwoSlopeNorm(vcenter=0, vmin=-dmax, vmax=dmax)

    # ---- figure layout
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.subplots_adjust(left=0.09, right=0.95, top=0.88, bottom=0.06, wspace=0.00, hspace=0.50)
    fig.suptitle(f"{pol_lbl} — {year} (S2 and S3)", fontsize=18)

    for i, sc in enumerate(scenarios):
        scen = data[sc]["scen"]
        corr = data[sc]["corr"]

        # Concentration map
        sc_plot = fast_plot(
            axes[i, 0],
            scen,
            "conc",
            f"{sc.upper()} — Average Concentration",
            "viridis",
            conc_norm
        )

        # Delta map
        delta_plot = fast_plot(
            axes[i, 1],
            corr,
            "delta_conc",
            f"{sc.upper()} — Δ Concentration",
            "coolwarm",
            delta_norm
        )

        # ---- Proper publication-style colorbars
        cbar_ax1 = axes[i, 0].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb1 = plt.colorbar(sc_plot, cax=cbar_ax1)
        cb1.set_label("Mean Conc (µg/m³)", fontsize=12)
        cb1.ax.tick_params(labelsize=14)

        cbar_ax2 = axes[i, 1].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb2 = plt.colorbar(delta_plot, cax=cbar_ax2)
        cb2.set_label("Δ Conc (µg/m³)", fontsize=14)
        cb2.ax.tick_params(labelsize=12)

    # ---- Save: publication-ready
    out_path = os.path.join(OUTPUT_DIR, f"absolute_mean_{pol_lbl}_{year}_S2_and_S3.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)

    logging.info(f"Saved: {out_path}")


# -------------------------
# Main driver
# -------------------------
if __name__ == "__main__":
    for pol in pollutants:
        for year in years:
            plot_maps_for_pollutant_year(pol, year)

    logging.info("✅ All maps generated with absolute mean formatting.")


2026-01-18 19:55:14,668 - INFO - Plotting PM2.5 - 2030
2026-01-18 19:55:14,669 - INFO -   Loading S2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS2_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:55:15,084 - INFO -   Loading S3


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS3_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:55:19,522 - INFO - Saved: data/2-output-data/maps_CHIMERE\absolute_mean_PM2.5_2030_S2_and_S3.png
2026-01-18 19:55:19,553 - INFO - Plotting PM2.5 - 2050
2026-01-18 19:55:19,553 - INFO -   Loading S2


...

2026-01-18 19:55:52,261 - INFO - Saved: data/2-output-data/maps_CHIMERE\absolute_mean_NO2_2030_S2_and_S3.png
2026-01-18 19:55:52,551 - INFO - Plotting NO2 - 2050
2026-01-18 19:55:52,551 - INFO -   Loading S2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS2_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:55:55,057 - INFO -   Loading S3


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS3_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


2026-01-18 19:56:19,389 - INFO - Saved: data/2-output-data/maps_CHIMERE\absolute_mean_NO2_2050_S2_and_S3.png
2026-01-18 19:56:19,647 - INFO - ✅ All maps generated with absolute mean formatting.


In [ ]:
#-----------------------------------------------------------
# Codes to plot CHIMERE maps for PM2.5 and NO2 concentrations (Relative Change)
#---------------------------------------------------------
import os
import logging
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

scenarios = ["s2", "s3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

OUTPUT_DIR = "data/2-output-data/maps_CHIMERE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 13,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11
})


def pretty_pol(pol):
    return "PM2.5" if pol == "ug_PM25_RH50" else "NO2"


def fast_plot(ax, gdf, col, title, cmap, norm, fmt_mean="{:.2f}"):
    xs = gdf.geometry.x.values
    ys = gdf.geometry.y.values
    if len(xs) == 0 or len(ys) == 0:
        xmin, xmax, ymin, ymax = 0, 1, 0, 1
    else:
        xmin, xmax = np.nanmin(xs), np.nanmax(xs)
        ymin, ymax = np.nanmin(ys), np.nanmax(ys)
        pad_x = max((xmax - xmin) * 0.02, 1e-6)
        pad_y = max((ymax - ymin) * 0.02, 1e-6)
        xmin, xmax = xmin - pad_x, xmax + pad_x
        ymin, ymax = ymin - pad_y, ymax + pad_y

    sc = ax.scatter(
        xs, ys,
        c=gdf[col].values,
        s=7,
        cmap=cmap,
        norm=norm,
        linewidths=0,
        alpha=0.95,
        rasterized=True
    )

    mean_val = np.nanmean(gdf[col].values)
    ax.set_title(f"{title}\nMean = {fmt_mean.format(mean_val)}", pad=6)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.tick_params(axis='both', which='major', length=4, width=0.8, direction='out')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    return sc


def _ensure_conc_col(df, label_for_log):
    if "conc" in df.columns:
        return df
    for alt in ["meanconc", "conc19"]:
        if alt in df.columns:
            df = df.copy()
            df["conc"] = df[alt]
            logging.warning(f"{label_for_log}: using '{alt}' as 'conc'")
            return df
    logging.error(f"{label_for_log} data missing 'conc' column")
    return None


def plot_maps_for_pollutant_year(pol, year):
    pol_lbl = pretty_pol(pol)
    logging.info(f"Plotting {pol_lbl} - {year}")

    # Load 2019 base once
    base19 = coordo_ineris_chimere(pol, year="2019").copy()
    base19 = _ensure_conc_col(base19, "Base 2019")
    if base19 is None:
        return

    # Load scenario data and keep structures
    data = {}
    for sc in scenarios:
        logging.info(f"  Loading {sc.upper()} {year}")
        scen = coordo_chimere(pol, year=year, SC=sc.upper()).copy()
        scen = _ensure_conc_col(scen, f"Scenario {sc.upper()} {year}")
        if scen is None:
            return
        data[sc] = {"scen": scen}

    # Compute conc color range across 2019 and all scen concs for this year
    conc_arrays = [base19["conc"].values]
    for sc in scenarios:
        conc_arrays.append(data[sc]["scen"]["conc"].values)
    all_conc = np.concatenate(conc_arrays)
    conc_vmin = np.nanmin(all_conc)
    conc_vmax = np.nanmax(all_conc)
    conc_norm = Normalize(vmin=conc_vmin, vmax=conc_vmax)

    # Prepare figure: 2 rows (scenarios) x 3 cols (2019, year, % change)
    fig, axes = plt.subplots(2, 3, figsize=(17.5, 9.5))
    fig.subplots_adjust(left=0.06, right=0.98, top=0.90, bottom=0.07, wspace=0.32, hspace=0.36)
    fig.suptitle(f"{pol_lbl} — {year} (Left: 2019 mean; middle: {year} mean; right: % change vs 2019)", fontsize=16)

    # For percent change norm choose symmetric range based on max abs % across scenarios
    pct_max = 0.0
    for sc in scenarios:
        scen = data[sc]["scen"]
        base_vals = base19["conc"].values
        delta = scen["conc"].values - base_vals
        pct = np.where(base_vals == 0, np.nan, 100.0 * delta / base_vals)
        pct_max = max(pct_max, np.nanmax(np.abs(pct)))
    if pct_max == 0:
        pct_max = 1.0
    pct_norm = TwoSlopeNorm(vcenter=0, vmin=-pct_max, vmax=pct_max)

    # Plot each scenario row
    for i, sc in enumerate(scenarios):
        scen = data[sc]["scen"]
        # Column 0: 2019
        sc_plot = fast_plot(axes[i, 0], base19, "conc", f"{sc.upper()} — 2019 mean conc", cmap="viridis",
                            norm=conc_norm)
        # Column 1: target year mean conc
        sc2_plot = fast_plot(axes[i, 1], scen, "conc", f"{sc.upper()} — {year} mean conc", cmap="viridis",
                             norm=conc_norm)
        # Column 2: percent change
        base_vals = base19["conc"].values
        delta = scen["conc"].values - base_vals
        pct = np.where(base_vals == 0, np.nan, 100.0 * delta / base_vals)
        scen_pct = scen.copy()
        scen_pct["pct_change"] = pct
        pct_plot = fast_plot(axes[i, 2], scen_pct, "pct_change", f"{sc.upper()} — % change vs 2019", cmap="coolwarm",
                             norm=pct_norm, fmt_mean="{:.2f}")

        # Per-plot colorbars placed next to each map
        for ax, mappable, label in [
            (axes[i, 0], sc_plot, "Mean Conc (µg/m³)"),
            (axes[i, 1], sc2_plot, "Mean Conc (µg/m³)"),
            (axes[i, 2], pct_plot, "% change")
        ]:
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="3.5%", pad=0.10)
            cb = plt.colorbar(mappable, cax=cax)
            cb.set_label(label, fontsize=11)
            cb.ax.tick_params(labelsize=10)

    out_path = os.path.join(OUTPUT_DIR, f"{pol_lbl}_{year}_2019_vs_{year}_S2_S3.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.04)
    plt.close(fig)
    logging.info(f"Saved: {out_path}")


if __name__ == "__main__":
    for pol in pollutants:
        for year in years:
            plot_maps_for_pollutant_year(pol, year)
    logging.info("✅ All maps generated with publication-grade formatting.")
